# Hyperparameter Tuning

This notebook reads baseline CV results from unified_model_cv_results.csv, selects the top-3 models,
runs segment-wise 5-fold TimeSeriesSplit grid search, logs all combinations, and exports a compact paper table.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import importlib
import json
import sys
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

PROCESSING_DIR = Path.cwd().resolve()
if PROCESSING_DIR.name != 'processing':
    alt_dir = PROCESSING_DIR / 'src' / 'processing'
    if alt_dir.exists():
        PROCESSING_DIR = alt_dir
if str(PROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(PROCESSING_DIR))

import modeling_common
importlib.reload(modeling_common)

from modeling_common import (
    SEED,
    TARGET,
    build_model_registry,
    evaluate_estimator_timeseries,
    get_param_grids,
    get_segment_data,
    resolve_project_root,
    resolve_unified_summary_path,
    load_preprocessed_df,
)

np.random.seed(SEED)
print('Shared module loaded from:', PROCESSING_DIR / 'modeling_common.py')

Shared module loaded from: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\src\processing\modeling_common.py


In [ ]:
ROOT = resolve_project_root()
df, PREPROCESSED_PATH = load_preprocessed_df(ROOT, keep_target_only=True)
BASELINE_PATH = resolve_unified_summary_path(ROOT)
baseline_summary = pd.read_csv(BASELINE_PATH)

print('Loaded preprocessed data:', PREPROCESSED_PATH)
print('Loaded baseline summary:', BASELINE_PATH)
print('Data shape:', df.shape)
print('Baseline rows:', len(baseline_summary))

Loaded preprocessed data: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\tea_preprocessed.csv
Loaded baseline summary: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\unified_model_cv_results.csv
Data shape: (2886, 182)
Baseline rows: 32


In [ ]:
segment_data = get_segment_data(df, target=TARGET)

for seg, (sdf, feature_cols) in segment_data.items():
    print(f"{seg:<10} rows={len(sdf):4d} features={len(feature_cols):3d}")

High Grown rows= 516 features=167
Low Grown  rows=1348 features=167
Off-Grade  rows= 499 features=167
Dust       rows= 523 features=167


In [ ]:
 # Leakage audit for candidate leakage columns.
leak_cols = ['price_lo_lkr', 'price_hi_lkr', 'price_range_lkr', 'tier']
audit_rows = []

for seg, (sdf, feature_cols) in segment_data.items():
    feature_set = set(feature_cols)
    present_in_raw = {c: (c in sdf.columns) for c in leak_cols}
    present_in_features = {c: (c in feature_set) for c in leak_cols}
    leaked_cols = [c for c in leak_cols if present_in_features[c]]

    audit_rows.append({
        'Segment': seg,
        'n_features': len(feature_cols),
        'raw_has_all_4': all(present_in_raw.values()),
        'leak_cols_in_training_features': ', '.join(leaked_cols) if leaked_cols else 'NONE',
        'leakage_pass': len(leaked_cols) == 0,
    })

leakage_audit_df = pd.DataFrame(audit_rows)
display(leakage_audit_df)

if leakage_audit_df['leakage_pass'].all():
    print('Leakage audit PASS: none of the specified columns are in training features.')
else:
    print('Leakage audit FAIL: remove listed columns from training features before tuning.')

,Segment,n_features,raw_has_all_4,leak_cols_in_training_features,leakage_pass
0,High Grown,167,True,NONE,True
1,Low Grown,167,True,NONE,True
2,Off-Grade,167,True,NONE,True
3,Dust,167,True,NONE,True


Leakage audit PASS: none of the specified columns are in training features.


In [ ]:
PARAM_GRIDS = get_param_grids()
print('Parameter grids loaded for models:', list(PARAM_GRIDS.keys()))

Parameter grids loaded for models: ['Ridge', 'Lasso', 'ElasticNet', 'Random Forest', 'Gradient Boosting', 'SVR (RBF)', 'XGBoost', 'LightGBM']


In [ ]:
# Select top-3 models and run segment-wise grid search.
top3_models = (
    baseline_summary.groupby('Model', as_index=False)['RMSE']
    .mean()
    .sort_values('RMSE')
    .head(3)['Model']
    .tolist()
)
print('Top-3 models from T2.1:', top3_models)

base_models = build_model_registry(seed=SEED)
inner_cv = TimeSeriesSplit(n_splits=5)

all_grid_logs = []
best_rows = []

for seg_name, (sdf, feature_cols) in segment_data.items():
    print(f'\nTuning segment: {seg_name} | rows={len(sdf)} | features={len(feature_cols)}')
    if len(sdf) < 20:
        print(f'  Skipped {seg_name}: not enough rows for stable 5-fold tuning.')
        continue

    X = sdf[feature_cols].copy()
    y = sdf[TARGET].copy()
    segment_best = None

    for model_name in top3_models:
        if model_name not in PARAM_GRIDS:
            print(f'  Skipped {model_name}: no grid defined.')
            continue

        model_obj = base_models[model_name]
        param_grid = PARAM_GRIDS[model_name]

        gscv = GridSearchCV(
            estimator=model_obj,
            param_grid=param_grid,
            scoring='neg_root_mean_squared_error',
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            return_train_score=False,
        )
        gscv.fit(X, y)

        grid_log = pd.DataFrame(gscv.cv_results_)[['params', 'mean_test_score', 'std_test_score', 'rank_test_score']].copy()
        grid_log['Segment'] = seg_name
        grid_log['Model'] = model_name
        grid_log['mean_test_RMSE_log'] = -grid_log['mean_test_score']
        all_grid_logs.append(grid_log)

        fold_eval = evaluate_estimator_timeseries(
            sdf=sdf,
            feature_cols=feature_cols,
            estimator=gscv.best_estimator_,
            target=TARGET,
            k=5,
        )

        row = {
            'Segment': seg_name,
            'Model': model_name,
            'RMSE': fold_eval['RMSE'].mean(),
            'MAE': fold_eval['MAE'].mean(),
            'MAPE': fold_eval['MAPE'].mean(),
            'R2': fold_eval['R2'].mean(),
            'Selected_Hyperparameters': json.dumps(gscv.best_params_, sort_keys=True),
        }

        if (segment_best is None) or (row['RMSE'] < segment_best['RMSE']):
            segment_best = row

        print(f"  Done: {model_name} | best RMSE={row['RMSE']:.2f}")

    if segment_best is not None:
        best_rows.append(segment_best)

grid_logs_df = pd.concat(all_grid_logs, ignore_index=True) if all_grid_logs else pd.DataFrame()
tuned_best_df = pd.DataFrame(best_rows)

print('\nAll grid-search logs shape:', grid_logs_df.shape)
print('Best-per-segment table shape:', tuned_best_df.shape)
display(tuned_best_df.sort_values('RMSE'))

Top-3 models from T2.1: ['Gradient Boosting', 'XGBoost', 'Random Forest']

Tuning segment: High Grown | rows=516 | features=167
  Done: Gradient Boosting | best RMSE=17.31
  Done: XGBoost | best RMSE=23.76
  Done: Random Forest | best RMSE=27.80

Tuning segment: Low Grown | rows=1348 | features=167
  Done: Gradient Boosting | best RMSE=29.26
  Done: XGBoost | best RMSE=60.50
  Done: Random Forest | best RMSE=59.10

Tuning segment: Off-Grade | rows=499 | features=167
  Done: Gradient Boosting | best RMSE=11.49
  Done: XGBoost | best RMSE=19.88
  Done: Random Forest | best RMSE=18.90

Tuning segment: Dust | rows=523 | features=167
  Done: Gradient Boosting | best RMSE=16.06
  Done: XGBoost | best RMSE=22.48
  Done: Random Forest | best RMSE=20.24

All grid-search logs shape: (324, 7)
Best-per-segment table shape: (4, 7)


,Segment,Model,RMSE,MAE,MAPE,R2,Selected_Hyperparameters
2,Off-Grade,Gradient Boosting,11.485238,7.390936,1.079218,0.990054,"{""model__learning_rate"": 0.1, ""model__max_dept..."
3,Dust,Gradient Boosting,16.059939,10.116310,1.160942,0.983794,"{""model__learning_rate"": 0.1, ""model__max_dept..."
0,High Grown,Gradient Boosting,17.307148,11.730450,1.186913,0.994927,"{""model__learning_rate"": 0.1, ""model__max_dept..."
1,Low Grown,Gradient Boosting,29.264202,14.098979,0.744588,0.999054,"{""model__learning_rate"": 0.05, ""model__max_dep..."


In [ ]:
# Save full logs and compact table for the paper.
if grid_logs_df.empty or tuned_best_df.empty:
    print('No tuning outputs to save.')
else:
    out_dir = ROOT / 'data' / 'processed'
    out_dir.mkdir(parents=True, exist_ok=True)

    logs_path = out_dir / 'hyperparam_gridsearch_all_results.csv'
    compact_path = out_dir / 'hyperparam_best_per_segment_compact.csv'

    grid_logs_df.to_csv(logs_path, index=False)

    compact_table = (
        tuned_best_df[['Segment', 'Model', 'RMSE', 'MAE', 'Selected_Hyperparameters']]
        .sort_values('RMSE')
        .reset_index(drop=True)
    )
    compact_table.to_csv(compact_path, index=False)

    print('Saved full tuning log to:', logs_path)
    print('Saved compact paper table to:', compact_path)
    display(compact_table)

Saved full tuning log to: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\hyperparam_gridsearch_all_results.csv
Saved compact paper table to: C:\Users\senil\Documents\GitHub\data-analysis-for-tea-industry\data\processed\hyperparam_best_per_segment_compact.csv


,Segment,Model,RMSE,MAE,Selected_Hyperparameters
0,Off-Grade,Gradient Boosting,11.485238,7.390936,"{""model__learning_rate"": 0.1, ""model__max_dept..."
1,Dust,Gradient Boosting,16.059939,10.116310,"{""model__learning_rate"": 0.1, ""model__max_dept..."
2,High Grown,Gradient Boosting,17.307148,11.730450,"{""model__learning_rate"": 0.1, ""model__max_dept..."
3,Low Grown,Gradient Boosting,29.264202,14.098979,"{""model__learning_rate"": 0.05, ""model__max_dep..."
